# Discrete PySPIDER on the flocking simulation

Run all cells with the PySPIDER Conda interpreter. The notebook generates a trajectory with `../particles.py` or loads a saved trajectory, then evaluates discrete libraries and runs sparse model discovery.

The default settings provide a short initial run. [02_quadrature_comparison.ipynb](02_quadrature_comparison.ipynb) compares quadrature schemes on the saved data.

Reading: [dissertation](../../DG_Dissertation.pdf), §2.4 (pp.38–48) for the discrete construction and §5.2 (pp.107–119) for its compressible-fluid application; [discrete API tutorial](../../tutorials/02_Discrete.ipynb).


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, subprocess, runpy, uuid
from datetime import datetime, timezone

# Resolve paths when launched from dynamics, flocking, or repository root.
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/PySPIDER').is_dir() and (p / 'flocking/particles.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Launch this notebook from within the PySPIDER repository.')
DYNAMICS = REPO / 'flocking/dynamics'
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(DYNAMICS))
os.environ.setdefault('NUMBA_NUM_THREADS', '2')
import numpy as np
import scipy
import PySPIDER
from flocking_data import load_trajectories
from PySPIDER.commons.library import Observable
from PySPIDER.discrete.process_library_terms import SRDataset
from PySPIDER.commons.identify_models import interleave_identify
from PySPIDER.commons.sparse_reg_bf import Scaler, Initializer, ModelIterator, Residual, Threshold
assert Path(PySPIDER.__file__).resolve().is_relative_to(REPO / 'src')
print('Interpreter:', sys.executable)
print('PySPIDER:', PySPIDER.__file__)
print('NumPy:', np.__version__, 'SciPy:', scipy.__version__)


## Run settings

`TRAJECTORY_PATH = None` generates data using the simulator's current parameters. To load a saved `.npy` file, enter its path relative to this notebook's directory and set `EXISTING_DT`, `EXISTING_L`, and `EXISTING_PERIODIC_X` to the simulation settings. The array stores step indices; the adapter uses `EXISTING_DT` to compute the interval between saved frames.

The initial scalar/vector libraries allow complexity ≤4 and at most two coarse-grained factors per term. `MAX_DT=1` and `MAX_DX=2` limit derivative order on each factor; products may differentiate both factors. `MAX_COMPLEXITY` and `INCLUDE_RANK2` expand the search. `KERNEL_SIGMA` sets the PySPIDER kernel scale; the visualization script's `--h-factor` uses a different kernel convention.


In [ ]:
TRAJECTORY_PATH = None  # e.g. '../trajectories.npy'; None generates a fresh run
EXISTING_DT = 0.01
EXISTING_L = 126.0
EXISTING_PERIODIC_X = False
SIMULATION_T = 12.0
SAVE_EVERY = 20

DOMAIN_SEED = 1
KERNEL_SIGMA = 4.0
CG_RES = 1.0  # coarse-grained samples per physical length unit
DOMAIN_WIDTH = 24.0
DOMAIN_FRAMES = 25
N_DOMAINS = 12
WEIGHT_M = 6
WEIGHT_QMAX = 1  # 8 scalar weight functions in x,y,t
MAX_COMPLEXITY = 4
MAX_RHO = 2
MAX_DT = 1
MAX_DX = 2
INCLUDE_RANK2 = False
L_SCALE = 8.0  # explicit normalization scales, not measured correlations
T_SCALE = 2.0
DISCOVERY_THRESHOLD = 2e-3  # exploratory normalized training-residual cutoff

RUN_DIR = DYNAMICS / 'runs' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ') + '_' + uuid.uuid4().hex[:8])
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('New run directory:', RUN_DIR)


## Generate or select trajectories

Each execution creates a new output directory. Generated trajectories use the simulator's seeded initialization, constant-speed projected-force update, close-encounter cap, and wall reflections.


In [ ]:
particles_script = REPO / 'flocking/particles.py'
if TRAJECTORY_PATH is None:
    if not np.isfinite(SIMULATION_T) or SIMULATION_T <= 0 or SAVE_EVERY < 1:
        raise ValueError('Use a positive simulation duration and saving interval.')
    simulation = runpy.run_path(str(particles_script), run_name='flocking_simulation')
    trajectory_path = RUN_DIR / 'trajectories.npy'
    simulation['run_to_T'](SIMULATION_T, SAVE_EVERY, str(trajectory_path))
    simulation_dt = float(simulation['dt'])
    box_side = float(simulation['L'])
    periodic_x = bool(simulation['PERIODIC_X'])
    simulation_metadata = {k: simulation[k] for k in
        ['N', 'L', 'a', 'v', 'm', 'beta', 'eps', 'dt', 'seed', 'PERIODIC_X', 'rc', 'r_min']}
    simulation_metadata.update(source='generated by supplied particles.py', requested_T=SIMULATION_T, save_every=SAVE_EVERY)
else:
    trajectory_path = Path(TRAJECTORY_PATH).expanduser()
    if not trajectory_path.is_absolute():
        trajectory_path = DYNAMICS / trajectory_path
    trajectory_path = trajectory_path.resolve(strict=True)
    simulation_dt, box_side, periodic_x = float(EXISTING_DT), float(EXISTING_L), bool(EXISTING_PERIODIC_X)
    simulation_metadata = dict(source='existing trajectory; settings supplied by notebook user',
                               dt=simulation_dt, L=box_side, PERIODIC_X=periodic_x)
simulation_metadata['particles_script_sha256_at_run'] = hashlib.sha256(particles_script.read_bytes()).hexdigest()
(RUN_DIR / 'simulation.json').write_text(json.dumps(simulation_metadata, indent=2) + '\n')
print('Trajectory:', trajectory_path)
print(json.dumps(simulation_metadata, indent=2))


## Validate and reshape the particle data

Input rows are `[step, index, x, y, vx, vy]`, ordered by frame and particle index. The adapter converts them to `(particle, component, saved time)` arrays for discrete `SRDataset`. PySPIDER coarse-grains these trajectories during library evaluation; `coarse_grain.py` produces separate visualization fields.

The simulator always saves its final step. If the last saving interval is shorter than the others, the adapter drops that frame and records its step index. It rejects other irregular sampling and uses the retained samples directly.


In [ ]:
trajectory = load_trajectories(trajectory_path, simulation_dt=simulation_dt, box_side=box_side)
positions, velocities = trajectory['positions'], trajectory['velocities']
times, deltat = trajectory['times'], trajectory['deltat']
N, _, nt = positions.shape
if nt < DOMAIN_FRAMES:
    raise ValueError(f'{nt} saved frames are insufficient for DOMAIN_FRAMES={DOMAIN_FRAMES}. Generate longer data or reduce the domain.')
speed = np.linalg.norm(velocities, axis=1)
order = np.linalg.norm(velocities.mean(axis=0), axis=0) / speed.mean(axis=0)
print('Input shape (particles, components, times):', positions.shape)
print('Saved-frame dt:', deltat, '; time range:', (float(times[0]), float(times[-1])))
print('Dropped final step:', trajectory['dropped_final_step'])
print('Speed range:', (float(speed.min()), float(speed.max())))
print('Polar order, first / last:', float(order[0]), float(order[-1]))


## Build and evaluate discrete libraries

Discrete PySPIDER evaluates float64 fields on a lattice with spacing `1 / CG_RES`. Density is divided by the mean number density. Integration domains have padding of `4 * KERNEL_SIGMA`, which contains the default quartic kernel's support radius `sqrt(11) * KERNEL_SIGMA`.

`wrap=False` and interior padding keep kernel supports away from the reflecting walls. PySPIDER's `wrap=True` wraps both axes, while the simulator permits periodicity only in x. The calculation uses unique domains, serial matrix evaluation, and cached density fields for normalization. Omitted quadrature options select trapezoidal integration on all axes.


In [ ]:
padding = 4 * KERNEL_SIGMA
if KERNEL_SIGMA <= 0 or CG_RES <= 0 or DOMAIN_WIDTH <= 0 or DOMAIN_WIDTH + 2 * padding > box_side:
    raise ValueError('The kernel padding and spatial integration domain must fit inside the box.')
if DOMAIN_FRAMES < 8 or int(DOMAIN_WIDTH * CG_RES) < 8 or N_DOMAINS < 1:
    raise ValueError('Use at least 8 samples per integration axis and a positive domain count.')
if L_SCALE <= 0 or T_SCALE <= 0:
    raise ValueError('Normalization length and time scales must be positive.')
np.random.seed(DOMAIN_SEED)
irreps = SRDataset.all_rank2_irreps() if INCLUDE_RANK2 else SRDataset.all_rank2_irreps()[:2]
srd = SRDataset(world_size=np.array([box_side, box_side, nt]),
                data_dict={'v': velocities}, particle_pos=positions,
                observables=[Observable(string='v', rank=1)], irreps=irreps,
                kernel_sigma=KERNEL_SIGMA, cg_res=CG_RES, deltat=deltat,
                rho_scale=N / box_side**2, time_sigma=0, wrap=False,
                cache_primes=True, cleanup_cache=False)
srd.make_libraries(max_complexity=MAX_COMPLEXITY, max_rho=MAX_RHO, max_dt=MAX_DT, max_dx=MAX_DX)
srd.make_domains(ndomains=N_DOMAINS, domain_size=[DOMAIN_WIDTH, DOMAIN_WIDTH, DOMAIN_FRAMES], pad=padding)
if len(set(srd.domains)) != N_DOMAINS:
    raise ValueError('Duplicate sampled domains; change DOMAIN_SEED or reduce N_DOMAINS.')
srd.make_weights(m=WEIGHT_M, qmax=WEIGHT_QMAX)
srd.set_LT_scale(L=L_SCALE, T=T_SCALE)
srd.make_library_matrices(debug=False, parallel=False)
matrix_arrays = {}
library_records = []
for i, irrep in enumerate(srd.irreps):
    lib = srd.libs[irrep]
    assert np.isfinite(lib.Q).all(), f'Nonfinite matrix for {irrep}'
    assert lib.Q.shape[0] > lib.Q.shape[1], 'Increase domains/weights for an overdetermined library.'
    assert np.all(np.isfinite(lib.col_weights)) and np.all(np.asarray(lib.col_weights) > 0), 'Degenerate normalization scale'
    matrix_arrays[f'Q_{i}'] = lib.Q
    matrix_arrays[f'col_weights_{i}'] = np.asarray(lib.col_weights)
    library_records.append(dict(key=f'Q_{i}', irrep=str(irrep), shape=list(lib.Q.shape), terms=[str(t) for t in lib.terms]))
    print(str(irrep), 'matrix shape:', lib.Q.shape)
    print('Terms:', ', '.join(str(t) for t in lib.terms))
np.savez_compressed(RUN_DIR / 'library_matrices.npz', **matrix_arrays)
(RUN_DIR / 'libraries.json').write_text(json.dumps(library_records, indent=2, ensure_ascii=False) + '\n')


## Sparse model discovery

Regression uses physical column scales, unit row normalization, truncated inverse iteration, and backward/forward passes. All rows enter the fit. `DISCOVERY_THRESHOLD` controls the accepted training residual; this initial run uses 0.002.

Interpretation depends on the simulator's constraints: fixed speed produces exact moment identities, and force projection changes the momentum balance. Assessment of the inferred dynamics requires independent trajectories and checks of timestep, sampling, kernel scale, spatial resolution, domains, and library choice.


In [ ]:
reg_opts_list = []
for irrep in srd.irreps:
    lib = srd.libs[irrep]
    k = min(10, len(lib.terms))
    reg_opts_list.append(dict(
        scaler=Scaler(sub_inds=None, char_sizes=lib.col_weights, row_norms=None, unit_rows=True, train_fraction=1),
        initializer=Initializer(method='power', start_k=k),
        model_iterator=ModelIterator(max_k=k, backward_forward=True, max_passes=3),
        residual=Residual(residual_type='matrix_relative'),
        threshold=Threshold(threshold_type='jump', gamma=1.5, delta=1e-8),
        verbose=False))
np.random.seed(DOMAIN_SEED)
equations, residuals, regression_results, derived_equations, excluded_terms = interleave_identify(
    [srd.libs[i] for i in srd.irreps], reg_opts_list,
    threshold=DISCOVERY_THRESHOLD, max_equations=12, report_accuracy=True)
equation_records = [dict(equation=eq.pstr(), residual=float(res),
                         terms=[str(t) for t in eq.terms], coefficients=[float(c) for c in eq.coeffs])
                    for eq, res in zip(equations, residuals)]
print('Number of discovered relations:', len(equation_records))
for record in equation_records:
    print(record['equation'], '; normalized training residual =', record['residual'])


## Save reproducibility record

Each run saves settings, integration-domain corners, matrix columns and scales, and learned relations. NumPy archives and JSON records hold the results. For existing trajectories, the record includes the source path and checksum. Source hashes identify the local implementation alongside its Git revision.


In [ ]:
git_head = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, capture_output=True, text=True, check=True).stdout.strip()
source_paths = [particles_script, DYNAMICS / 'flocking_data.py', DYNAMICS / '01_flocking_discovery.ipynb',
                REPO / 'src/PySPIDER/discrete/process_library_terms.py',
                REPO / 'src/PySPIDER/discrete/coarse_grain_utils.py',
                REPO / 'src/PySPIDER/commons/identify_models.py',
                REPO / 'src/PySPIDER/commons/sparse_reg_bf.py',
                REPO / 'src/PySPIDER/commons/quadrature_schemes.py']
settings = {k: globals()[k] for k in ['DOMAIN_SEED', 'KERNEL_SIGMA', 'CG_RES', 'DOMAIN_WIDTH',
    'DOMAIN_FRAMES', 'N_DOMAINS', 'WEIGHT_M', 'WEIGHT_QMAX', 'MAX_COMPLEXITY', 'MAX_RHO',
    'MAX_DT', 'MAX_DX', 'INCLUDE_RANK2', 'L_SCALE', 'T_SCALE', 'DISCOVERY_THRESHOLD']}
run_summary = dict(
    created_utc=datetime.now(timezone.utc).isoformat(), git_head=git_head,
    interpreter=sys.executable, numpy_version=np.__version__, scipy_version=scipy.__version__,
    trajectory_path=str(trajectory_path), trajectory_sha256=hashlib.sha256(trajectory_path.read_bytes()).hexdigest(),
    simulation=simulation_metadata, settings=settings, particle_count=N, snapshots_used=nt,
    saved_frame_dt=deltat, dropped_final_step=trajectory['dropped_final_step'],
    matrix_shapes=[lib['shape'] for lib in library_records], libraries=library_records,
    domains=[dict(min_corner=[int(x) for x in d.min_corner], max_corner=[int(x) for x in d.max_corner]) for d in srd.domains],
    source_sha256={str(p.relative_to(REPO)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_paths},
    equations=equation_records, validation='Training fits only; no independent scientific validation.')
(RUN_DIR / 'results.json').write_text(json.dumps(run_summary, indent=2, ensure_ascii=False, allow_nan=False) + '\n')
print('Saved:', RUN_DIR / 'results.json')
print('Saved:', RUN_DIR / 'library_matrices.npz')
print('Saved:', RUN_DIR / 'libraries.json')
print('Saved:', RUN_DIR / 'simulation.json')
